# Preprocessing correctness & quality audit

Checks `utils/preprocessing.py` and the adapters in `preprocessing/` **against the raw data**
(`../data/raw`, the source of truth). Nothing here reads `../data/processed` — every image is
re-read from source and pushed through the real pipeline objects.

Sections:

1. Setup and check harness
2. Image primitives (`normalize_to_uint16`, `pad_to_largest_dim`, resize, mask ops)
3. Image ↔ segmentation geometry agreement
4. Readers on real raw files, one per format
5. BI-RADS lexicon
6. Descriptor mappers: closed vocabulary + idempotence
7. Location mapper (clock ↔ quadrant)
8. Mapping coverage against the raw metadata of every dataset
9. Mask alignment on real image/mask pairs
10. OASBUD: RF demodulation axis
11. EMBED: `match_level` filter and ROI coordinate order
12. LA-Breast: ROI rectangle
13. Report JSON construction
14. Summary

## 1. Setup and check harness

In [ ]:
import json
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

# The adapters hard-code paths relative to the repo root, e.g. "../data/raw/...".
RAW = (REPO_ROOT.parent / "data" / "raw").resolve()
print("repo:", REPO_ROOT)
print("raw :", RAW, "exists:", RAW.exists())
%matplotlib inline

In [ ]:
RESULTS = []


def check(section, name, passed, detail=""):
    RESULTS.append(
        {"section": section, "check": name, "status": "PASS" if passed else "FAIL", "detail": str(detail)}
    )
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f"  --  {detail}" if detail else ""))
    return passed


def note(section, name, detail):
    RESULTS.append({"section": section, "check": name, "status": "INFO", "detail": str(detail)})
    print(f"[INFO] {name}  --  {detail}")


def describe(a):
    a = np.asarray(a)
    return f"shape={a.shape} dtype={a.dtype} min={a.min()} max={a.max()}"


def show(ax, img, mask=None, title=""):
    ax.imshow(np.asarray(img), cmap="gray")
    m = None if mask is None else np.asarray(mask)
    if m is not None and m.max() > 0:
        ax.contour(m.astype(float), levels=[0.5], colors="r", linewidths=1.2)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

## 2. Image primitives

`normalize_to_uint16`, `pad_to_largest_dim`, `resize_breast_image`, `binarize_mask`, `resize_mask`,
`create_segmentation_mask`, `draw_rectangle_on_image`.

In [ ]:
from utils.preprocessing import (
    binarize_mask,
    create_segmentation_mask,
    draw_rectangle_on_image,
    normalize_to_uint16,
    pad_to_largest_dim,
    resize_breast_image,
    resize_mask,
)

S = "2. primitives"

# --- normalize_to_uint16 --------------------------------------------------
ramp = np.linspace(-1000, 3000, 256, dtype=np.float32).reshape(16, 16)
n = normalize_to_uint16(ramp)
check(S, "normalize_to_uint16 spans the full uint16 range", (n.min(), n.max()) == (0, 65535), describe(n))
check(S, "normalize_to_uint16 is monotone", np.all(np.diff(n.ravel()) >= 0))
check(S, "normalize_to_uint16 dtype is uint16", n.dtype == np.uint16)

flat = np.full((8, 8), 7.0, dtype=np.float32)
check(S, "constant image -> all zeros (no divide-by-zero)", normalize_to_uint16(flat).max() == 0)

nan_img = np.array([[np.nan, 0.0], [1.0, 2.0]], dtype=np.float32)
out = normalize_to_uint16(nan_img)
check(S, "NaN pixels collapse to 0 rather than propagating", np.isfinite(out).all() and out[0, 0] == 0, describe(out))

# explicit shared range (used by MAMA-MIA across contrast phases)
a = normalize_to_uint16(np.array([[0.0, 50.0]]), value_range=(0.0, 100.0))
check(S, "explicit value_range is honoured (50/100 -> mid grey)", abs(int(a[0, 1]) - 32768) <= 1, a.tolist())

# clipping outside an explicit range
b = normalize_to_uint16(np.array([[-10.0, 200.0]]), value_range=(0.0, 100.0))
check(S, "values outside value_range clip to the ends", (b[0, 0], b[0, 1]) == (0, 65535), b.tolist())

In [ ]:
# --- pad_to_largest_dim ---------------------------------------------------
img = np.zeros((100, 40), dtype=np.uint16)
img[10:20, 5:15] = 1000
p = pad_to_largest_dim(img)
check(S, "pad_to_largest_dim yields a square frame", p.shape == (100, 100), p.shape)
check(S, "padding preserves total signal (constant border = 0)", p.sum() == img.sum())

ys, xs = np.nonzero(p)
ys0, xs0 = np.nonzero(img)
check(
    S,
    "padding is centred (offset = (biggest-side - side)//2)",
    ys.min() == ys0.min() and xs.min() == xs0.min() + (100 - 40) // 2,
    f"row offset={ys.min() - ys0.min()}, col offset={xs.min() - xs0.min()} (expected 0, 30)",
)

# aspect ratio: pad-then-resize must not distort
tall = np.zeros((400, 100), dtype=np.uint16)
tall[100:300, 40:60] = 5000            # a 200x20 bar, aspect 10:1
proc = resize_breast_image(pad_to_largest_dim(tall), (512, 512))
ys, xs = np.nonzero(proc)
aspect = (ys.max() - ys.min() + 1) / (xs.max() - xs.min() + 1)
check(S, "pad+resize preserves aspect ratio (10:1 bar stays ~10:1)", abs(aspect - 10) / 10 < 0.05, f"aspect={aspect:.2f}")

In [ ]:
# --- mask ops -------------------------------------------------------------
m = np.array([[0, 3], [0, 255]], dtype=np.uint16)
check(S, "binarize_mask maps any non-zero to 1", binarize_mask(m).tolist() == [[0, 1], [0, 1]])
check(S, "binarize_mask returns uint8", binarize_mask(m).dtype == np.uint8)

big = np.zeros((100, 100), dtype=np.uint8)
big[40:60, 40:60] = 1
rm = resize_mask(big, (512, 512))
check(S, "resize_mask stays binary (nearest-neighbour)", set(np.unique(rm)) <= {0, 1}, np.unique(rm).tolist())
check(S, "resize_mask preserves area fraction", abs(rm.mean() - big.mean()) < 1e-3, f"{big.mean():.4f} -> {rm.mean():.4f}")

# --- create_segmentation_mask: box order is (ymin, xmin, ymax, xmax) ------
mask = create_segmentation_mask(height=100, width=200, rois=((10, 20, 30, 60),))
ys, xs = np.nonzero(mask)
check(
    S,
    "create_segmentation_mask reads boxes as (ymin, xmin, ymax, xmax)",
    (ys.min(), ys.max(), xs.min(), xs.max()) == (10, 29, 20, 59),
    f"rows {ys.min()}..{ys.max()}, cols {xs.min()}..{xs.max()}",
)
check(S, "nested ROI lists are flattened", create_segmentation_mask(50, 50, (((1, 1, 5, 5),),)).sum() == 16)
check(S, "out-of-frame ROI is dropped, not clipped into a corner", create_segmentation_mask(50, 50, ((80, 80, 90, 90),)).sum() == 0)

# --- draw_rectangle_on_image is PURE (returns a copy) ---------------------
canvas = np.zeros((64, 64), dtype=np.uint8)
draw_rectangle_on_image(canvas, 10, 10, 20, 20)          # return value discarded
in_place = canvas.max()
returned = draw_rectangle_on_image(canvas, 10, 10, 20, 20).max()
check(
    S,
    "draw_rectangle_on_image does NOT mutate its argument",
    in_place == 0 and returned > 0,
    f"argument.max()={in_place} (unchanged), return.max()={returned}  <-- callers MUST use the return value",
)

## 3. Image ↔ segmentation geometry agreement

The image pipeline is `pad_to_largest_dim -> resize(512,512)`; the mask pipeline is
`binarize -> pad_to_largest_dim -> resize_mask(512,512)`. A lesion must land on the same
pixels in both outputs, otherwise every mask in the corpus is offset.

In [ ]:
from dtos.dataset_preprocessing_config import RESIZE_DIMS, ImagePreprocessingConfig

S = "3. geometry"
cfg = ImagePreprocessingConfig()

rng = np.random.default_rng(0)
for shape in [(2457, 1890), (512, 512), (1824, 510), (300, 900)]:
    img = rng.integers(0, 4000, size=shape, dtype=np.uint16)
    gt = np.zeros(shape, dtype=np.uint8)
    y0, x0 = shape[0] // 3, shape[1] // 4
    h, w = max(4, shape[0] // 10), max(4, shape[1] // 10)
    gt[y0 : y0 + h, x0 : x0 + w] = 1
    img[gt > 0] = 65535                       # burn the lesion into the image

    pi, pm = img, gt
    for step in cfg.processing_pipeline:
        pi = step(pi)
    for step in cfg.segmentation_pipeline:
        pm = step(pm)

    # Compare centroids rather than IoU: a hard resize blurs the lesion edge, so a few
    # boundary pixels always disagree. A shifted mask is what actually matters.
    bright = pi >= 60000
    cy_i, cx_i = np.argwhere(bright).mean(axis=0)
    cy_m, cx_m = np.argwhere(pm > 0).mean(axis=0)
    offset = float(np.hypot(cy_i - cy_m, cx_i - cx_m))
    area_ratio = (pm > 0).sum() / max(bright.sum(), 1)
    # <=1 output pixel is inherent: the image is resampled bilinearly and the mask
    # nearest-neighbour, so they round a sub-pixel edge differently.
    check(
        S,
        f"mask sits on the image content for a {shape} frame",
        offset <= 1.01 and 0.9 < area_ratio < 1.15 and pi.shape == RESIZE_DIMS == pm.shape,
        f"centroid offset={offset:.2f}px, mask/lesion area={area_ratio:.3f}, img{pi.shape} mask{pm.shape}",
    )

check(S, "processed images keep uint16", pi.dtype == np.uint16, pi.dtype)
check(S, "processed masks are binary", set(np.unique(pm)) <= {0, 1}, np.unique(pm).tolist())

In [ ]:
# What actually lands on disk: save_func casts BOTH images and masks to uint16.
import inspect

from utils.preprocessing import save_images_as_npy

note(
    S,
    "save_func dtype",
    f"{cfg.save_func} -- masks are stored as uint16 0/1, not uint8; "
    "downstream code must not assume a mask dtype of uint8.",
)
note(S, "resize dims", f"every image and mask is squashed to {RESIZE_DIMS} after square padding")

## 4. Readers on real raw files

One real file per reader/format. Every dataset that is missing on disk is skipped, not failed.

In [ ]:
from utils.preprocessing import (
    convert_dcm_image,
    process_us_from_mat,
    read_breast_image,
    read_mat_images,
    read_nii_gz_images,
    read_nii_gz_raw,
)

S = "4. readers"


def first(pattern, root=RAW):
    hits = sorted(Path(root).glob(pattern))
    return hits[0] if hits else None


samples = {
    "cbis-ddsm (jpg)": (first("cbis-ddsm/jpeg/*/*.jpg"), read_breast_image),
    "embed (png)": (first("embed/imgs/*.png"), read_breast_image),
    "la-breast (tiff)": (first("la-breast/imgs/d0/*/*.tiff"), read_breast_image),
    "breast-lesions-usg (png)": (first("breast-lesions-usg/*/case*.png"), read_breast_image),
    "BUSCoT (img)": (first("BUSCoT/BUS-Expert/**/*.jpg") or first("BUSCoT/BUS-Expert/**/*.png"), read_breast_image),
    "inbreast (dcm)": (first("inbreast/AllDICOMs/*.dcm"), convert_dcm_image),
    "tompei-cmmd (dcm)": (first("tompei-cmmd/cmmd/**/*.dcm"), convert_dcm_image),
    "breast-micro-calc (dcm)": (first("breast-micro-calc/imgs/**/*_recent.dcm")
                                or first("breast-micro-calc/imgs/**/*.dcm"), convert_dcm_image),
    "advanced-mri (dcm)": (first("advanced-mri-breast-lesions/Advanced-MRI-Breast-Lesions/**/1-01.dcm")
                           or first("advanced-mri-breast-lesions/Advanced-MRI-Breast-Lesions/**/*.dcm"), convert_dcm_image),
    "mama-mia (nii.gz)": (first("mama-mia/images/*/*.nii.gz"), read_nii_gz_images),
}

rows, gallery = [], []
for name, (path, reader) in samples.items():
    if path is None:
        rows.append({"source": name, "file": "MISSING", "result": "skipped"})
        continue
    arr = reader(str(path))
    if arr is None:
        rows.append({"source": name, "file": path.name, "result": "reader returned None"})
        check(S, f"{name} reads", False, str(path))
        continue
    a = np.asarray(arr)
    frame = a if a.ndim == 2 else a[a.shape[0] // 2]
    rows.append(
        {"source": name, "file": path.name, "result": describe(a), "2d frame": str(frame.shape)}
    )
    gallery.append((name, frame))
    check(S, f"{name} reads to uint16 with signal", a.dtype == np.uint16 and a.max() > a.min(), describe(a))

display(pd.DataFrame(rows))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, (name, frame) in zip(axes.ravel(), gallery):
    show(ax, frame, title=f"{name}\n{frame.shape}")
for ax in axes.ravel()[len(gallery) :]:
    ax.axis("off")
fig.suptitle("Raw readers -- one real file per format (before pad/resize)")
plt.tight_layout()
plt.show()

In [ ]:
# The same files after the real processing pipeline.
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for ax, (name, frame) in zip(axes.ravel(), gallery):
    out = frame
    for step in cfg.processing_pipeline:
        out = step(out)
    show(ax, out, title=f"{name}\n{out.shape} {out.dtype}")
    check("4. readers", f"{name} survives the pipeline", out.shape == RESIZE_DIMS and out.max() > 0, describe(out))
for ax in axes.ravel()[len(gallery) :]:
    ax.axis("off")
fig.suptitle("After pad_to_largest_dim -> resize(512, 512)")
plt.tight_layout()
plt.show()

In [ ]:
# DICOM photometric handling: MONOCHROME1 must come out with a dark background.
import pydicom as dicom

from utils.preprocessing import apply_dicom_photometric_transforms

S = "4. readers"
dcm_paths = [p for p in [samples[k][0] for k in samples if "dcm" in k] if p is not None]
rows = []
for p in dcm_paths:
    ds = dicom.dcmread(str(p))
    photometric = str(getattr(ds, "PhotometricInterpretation", "?"))
    out = convert_dcm_image(str(p))
    corner = float(np.median([out[0, 0], out[0, -1], out[-1, 0], out[-1, -1]]))
    rows.append(
        {
            "file": p.parts[-4] if len(p.parts) > 4 else p.name,
            "PhotometricInterpretation": photometric,
            "BitsStored": getattr(ds, "BitsStored", None),
            "median corner value": corner,
            "median overall": float(np.median(out)),
        }
    )
    check(
        S,
        f"{p.name}: background darker than tissue after photometric handling",
        corner <= float(np.median(out)),
        f"{photometric}: corners={corner:.0f} vs median={np.median(out):.0f}",
    )
display(pd.DataFrame(rows))

## 5. BI-RADS lexicon

In [ ]:
from utils.preprocessing import (
    BIRADS_CATEGORIES,
    BIRADS_MAPPING,
    MAMMOGRAPHY,
    MRI,
    ULTRASOUND,
    birads_category_number,
    collapse_known_biopsy_proven,
    get_proper_birads,
    map_birads,
)

S = "5. BI-RADS"

# every canonical category must survive a re-mapping (the adapters call
# get_proper_birads on an already-mapped value)
bad = [c for c in BIRADS_CATEGORIES if get_proper_birads(c, modality=MAMMOGRAPHY) != c]
check(S, "canonical categories round-trip under get_proper_birads (mammography)", not bad, bad)

# non-mammography collapses the 4A/4B/4C subdivision
sub = ["4a", "4b", "4c"]
collapsed = {m: {s: get_proper_birads(s, modality=m) for s in sub} for m in (ULTRASOUND, MRI)}
check(
    S,
    "4A/4B/4C collapse to '(4) suspicious' outside mammography",
    all(v == "(4) suspicious" for m in collapsed for v in collapsed[m].values()),
    collapsed[ULTRASOUND],
)
note(S, "granularity loss", "US/MRI sources that record 4A/4B/4C (e.g. BrEaST) lose the subdivision")

# numeric and free-text surfaces used by the adapters
surfaces = ["0", 0, 1, "1", 2.0, "3", "4A", "BI-RADS 4b", "bi rads 5", "6", "benign", "probably benign", "recall"]
mapped = [map_birads(s, source="audit") for s in surfaces]
tbl = pd.DataFrame(
    {
        "input": [repr(s) for s in surfaces],
        "mapped": mapped,
        "category number": [birads_category_number(m) for m in mapped],
    }
)
display(tbl)
check(S, "text and integer surfaces all map", all(m is not None for s, m in zip(surfaces, mapped) if not isinstance(s, float)))
check(
    S,
    "a float BI-RADS (2.0) maps like the integer 2",
    map_birads(2.0, source="audit") == map_birads(2, source="audit"),
    "a numeric column holding a NaN anywhere is read as float, so category 2 arrives as 2.0",
)
err = ""
try:
    nan_ok = birads_category_number(np.nan) is None
except Exception as e:
    nan_ok, err = False, e
check(S, "birads_category_number tolerates NaN as well as None", nan_ok, err if not nan_ok else "")

check(S, "BIRADS_MAPPING[1] is the negative category the adapters compare against",
      BIRADS_MAPPING[1] == "(1) negative", BIRADS_MAPPING[1])
check(S, "collapse_known_biopsy_proven folds 6 into 5",
      collapse_known_biopsy_proven(BIRADS_MAPPING[6]) == BIRADS_MAPPING[5], BIRADS_MAPPING[6])
check(S, "unmappable values return None instead of raising", map_birads("banana", source="audit") is None)

## 6. Descriptor mappers — closed vocabulary and idempotence

Every mapper must (a) return a value from its own declared vocabulary or `None`, and
(b) be idempotent: `f(f(x)) == f(x)`, because several adapters map a column and then
map it again downstream.

In [ ]:
from utils import preprocessing as P

S = "6. mappers"

MAPPERS = [
    ("breast composition (mg)", P.get_proper_breast_density, P.BREAST_COMPOSITION_ORDER, {}),
    ("fgt (mri)", P.get_proper_mri_fgt, P.MRI_FGT_ORDER, {}),
    ("tissue composition (us)", P.get_proper_tissue_composition, P.TISSUE_COMPOSITION_VALUES, {}),
    ("glandular tissue component", P.get_proper_glandular_tissue_component, P.GTC_ORDER, {}),
    ("bpe level", P.get_proper_bpe_level, P.BPE_LEVEL_ORDER, {}),
    ("bpe symmetry", P.get_proper_bpe_symmetry, P.BPE_SYMMETRY_VALUES, {}),
    ("mass shape", P.get_proper_mass_shape, P.MASS_SHAPE_ORDER, {}),
    ("mass margin (mg)", P.get_proper_mass_margin, P.MARGIN_ORDER[P.MAMMOGRAPHY], {"modality": P.MAMMOGRAPHY}),
    ("mass margin (us)", P.get_proper_mass_margin, P.MARGIN_ORDER[P.ULTRASOUND], {"modality": P.ULTRASOUND}),
    ("mass margin (mri)", P.get_proper_mass_margin, P.MARGIN_ORDER[P.MRI], {"modality": P.MRI}),
    ("mass density", P.get_proper_mass_density, P.MASS_DENSITY_ORDER, {}),
    ("orientation", P.get_proper_orientation, P.ORIENTATION_ORDER, {}),
    ("mass echo pattern", P.get_proper_mass_echo_pattern, P.MASS_ECHO_PATTERN_ORDER, {}),
    ("posterior features", P.get_proper_posterior_features, P.POSTERIOR_FEATURES_ORDER, {}),
    ("non-mass echo pattern", P.get_proper_non_mass_echo_pattern, P.NON_MASS_ECHO_PATTERN_ORDER, {}),
    ("nme internal enhancement", P.get_proper_nme_internal_enhancement, P.NME_INTERNAL_ENHANCEMENT_ORDER, {}),
    ("mass internal enhancement", P.get_proper_mass_internal_enhancement, P.MASS_INTERNAL_ENHANCEMENT_ORDER, {}),
    ("t2 signal", P.get_proper_t2_signal_intensity, P.T2_SIGNAL_VALUES, {}),
    ("early phase", P.get_proper_early_phase, P.EARLY_PHASE_ORDER, {}),
    ("delayed phase", P.get_proper_delayed_phase, P.DELAYED_PHASE_ORDER, {}),
    ("calc distribution", P.get_proper_calcification_distribution, P.CALCIFICATION_DISTRIBUTION_ORDER, {}),
    ("nm distribution (us)", P.get_proper_non_mass_distribution,
     P.NON_MASS_DISTRIBUTION_ORDER[P.ULTRASOUND], {"modality": P.ULTRASOUND}),
    ("nm distribution (mri)", P.get_proper_non_mass_distribution,
     P.NON_MASS_DISTRIBUTION_ORDER[P.MRI], {"modality": P.MRI}),
    ("vascularity", P.get_proper_vascularity, P.VASCULARITY_VALUES, {}),
    ("elasticity", P.get_proper_elasticity, P.ELASTICITY_ORDER, {}),
    ("echogenic feature", P.get_proper_echogenic_feature, P.ECHOGENIC_FEATURES, {}),
    ("exam view", P.get_proper_exam_view, (P.CRANIAL_CAUDAL, P.MEDIOLATERAL_OBLIQUE), {}),
]

rows = []
for name, fn, vocab, kw in MAPPERS:
    not_fixed = [v for v in vocab if fn(v, **kw) != v]
    nones = [v for v in vocab if fn(v, **kw) is None]
    na_ok = fn(None, **kw) is None and fn(np.nan, **kw) is None
    rows.append(
        {
            "mapper": name,
            "vocab size": len(vocab),
            "not idempotent on own vocab": not_fixed,
            "own term maps to None": nones,
            "None/NaN -> None": na_ok,
        }
    )
audit = pd.DataFrame(rows)
display(audit)

check(S, "every mapper is a fixed point on its own vocabulary",
      audit["not idempotent on own vocab"].apply(len).sum() == 0,
      audit.loc[audit["not idempotent on own vocab"].apply(len) > 0, "mapper"].tolist())
check(S, "every mapper returns None for None/NaN", audit["None/NaN -> None"].all())

In [ ]:
# Calcification type returns a (family, morphology) pair; check both halves.
S = "6. mappers"
rows = []
for term in P.SUSPICIOUS_CALCIFICATIONS + P.TYPICALLY_BENIGN_CALCIFICATIONS:
    fam, det = P.get_proper_calcification_type(term)
    rows.append({"input": term, "family": fam, "morphology": det, "round-trips": det == term})
calc = pd.DataFrame(rows)
display(calc)
check(S, "every v2025 calcification morphology round-trips", calc["round-trips"].all(),
      calc.loc[~calc["round-trips"], "input"].tolist())
check(S, "suspicious terms are filed under 'suspicious morphology'",
      set(calc[calc["input"].isin(P.SUSPICIOUS_CALCIFICATIONS)]["family"]) == {"suspicious morphology"})

# retired 5th-edition terms must land somewhere, not vanish
retired = {
    "milk of calcium": "layering",
    "punctate": "round",
    "dystrophic": "coarse",
    "popcorn-like": "coarse",
    "eggshell": "rim",
    "lucent-centered": "rim",
}
got = {k: P.get_proper_calcification_type(k)[1] for k in retired}
check(S, "retired 5th-ed calcification terms fold into their v2025 home", got == retired, got)

# compound values resolve to the most suspicious present term
compound = {
    "amorphous, fine pleomorphic": "fine pleomorphic",
    "round and punctate": "round",
    "coarse heterogeneous / fine linear branching": "fine linear or fine linear-branching",
}
got = {k: P.get_proper_calcification_type(k)[1] for k in compound}
check(S, "compound calcification values resolve to the most suspicious term", got == compound, got)

In [ ]:
# Asymmetry: "developing" was retired as a category but must survive as a flag.
S = "6. mappers"
rows = [{"input": v, "type": P.get_proper_asymmetry_type(v)[0], "is_developing": P.get_proper_asymmetry_type(v)[1]}
        for v in ["global asymmetry", "focal asymmetry", "asymmetry", "developing asymmetry",
                  "asymmetric breast tissue", "FAD", "developing"]]
display(pd.DataFrame(rows))
check(S, "'developing asymmetry' keeps its information in the is_developing flag",
      P.get_proper_asymmetry_type("developing asymmetry") == ("focal asymmetry", True))

# Mammography lost 'microlobulated' as a margin; it must move to the shape field.
shape, margin = P.migrate_mammography_microlobulated("oval", "microlobulated")
check(S, "microlobulated migrates from mammography margin to shape",
      (shape, margin) == ("lobulated", None), f"shape={shape!r} margin={margin!r}")
shape, margin = P.migrate_mammography_microlobulated("irregular", "microlobulated-spiculated")
check(S, "migration keeps a more suspicious shape and the residual margin",
      (shape, margin) == ("irregular", "spiculated"), f"shape={shape!r} margin={margin!r}")

# modality guards
try:
    P.get_proper_mass_margin("circumscribed", "xray")
    ok = False
except ValueError:
    ok = True
check(S, "mass margin rejects an unknown modality", ok)
check(S, "'obscured' is refused outside mammography",
      P.get_proper_mass_margin("obscured", P.ULTRASOUND) is None)
check(S, "'diffuse' is refused as an ultrasound non-mass distribution",
      P.get_proper_non_mass_distribution("diffuse", P.ULTRASOUND) is None)

## 7. Location mapper

In [ ]:
S = "7. location"

# right breast, patient perspective; left is the mirror
right = {h: P.map_clock_to_quadrant(h, "right") for h in range(1, 13)}
left = {h: P.map_clock_to_quadrant(h, "left") for h in range(1, 13)}
display(pd.DataFrame({"hour": list(range(1, 13)),
                      "right": [right[h] for h in range(1, 13)],
                      "left": [left[h] for h in range(1, 13)]}))

check(S, "10 o'clock right breast is upper outer", right[10] == "upper outer", right[10])
check(S, "10 o'clock left breast mirrors to upper inner", left[10] == "upper inner", left[10])
check(S, "12 and 6 stay on the midline (no quadrant claimed)",
      right[12] == "upper" and right[6] == "lower")
check(S, "a clock position without a single-breast laterality yields no quadrant",
      P.map_clock_to_quadrant(10, "bilateral") is None)

rows = []
for value, side in [("10 o'clock", "right"), ("upper outer quadrant", "left"), ("retroareolar", "right"),
                    ("3 o'clock, posterior third", "left"), ("UOQ", "right"), ("axillary tail", "left"),
                    ("entire region (covers the whole breast)", "right"), ("upper", "right")]:
    loc = P.get_proper_location(value, side)
    rows.append({"input": value, "side": side, **loc})
display(pd.DataFrame(rows))

loc = P.get_proper_location("3 o'clock, posterior third", "left")
check(S, "clock position and depth are kept side by side (not collapsed)",
      loc["clock_position"] == 3 and loc["depth"] == "posterior third" and loc["quadrant"] is None,
      loc)
check(S, "get_proper_location returns exactly the LesionLocation fields",
      set(P.get_proper_location("upper outer", "right")) ==
      {"laterality", "clock_position", "quadrant", "special_location", "coarse_location", "depth"})

from dtos.json_report_findings import LesionLocation
LesionLocation(**P.get_proper_location("10 o'clock", "right"))   # raises if the contract drifts
check(S, "the dict unpacks straight into LesionLocation", True)

## 8. Mapping coverage against the raw metadata

Runs the real mappers over the real columns of every dataset and reports the values that
come back `None`. Anything listed here is information present in the source that does not
reach the corpus.

In [ ]:
from collections import Counter

S = "8. coverage"


# Values a mapper is *meant* to reject: findings that exist in the source but are not the
# thing this particular field describes. They are reported separately from real gaps.
BY_DESIGN = {
    "architectural distortion", "questioned architectural distortion", "asymmetry",
    "global asymmetry", "focal asymmetry", "focal asymetry", "focal asymmetric density",
    "developing asymmetry", "asymmetric breast tissue", "lymph node", "lumph node",
    "intramammary lymph node", "axillary lymph node", "skin lesion", "generic",
    "asymmetric tubular structure", "solitary dilated duct", "non mass lesion",
    "**no assessment**", "lactating", "single", "unknown, unavailable or unreported",
    "not recorded", "patient declines", "whole area", "m-l", "0",
}


# Runs `fn` over the values an adapter would really pass it. `pre` replicates any
# transformation the adapter applies first (legend decode, int cast, ...); values are
# otherwise passed through unchanged, since stringifying them would change what the
# mapper sees (numpy 2.0 -> "2.0" misses lexicons keyed on "2").
def coverage(label, series, fn, pre=None, **kw):
    s = pd.Series(series).dropna()
    buckets, seen = {}, {}
    for v in s:
        key = str(v).strip()
        seen[key] = seen.get(key, 0) + 1
        if key in buckets:
            continue
        val = pre(v) if pre is not None else v
        try:
            r = fn(val, **kw) if val is not None else None
        except Exception as e:
            r = f"RAISED {type(e).__name__}: {e}"
        buckets[key] = r

    unmapped, by_design = {}, {}
    for key, r in buckets.items():
        if not (r is None or (isinstance(r, tuple) and r[0] is None) or str(r).startswith("RAISED")):
            continue
        (by_design if key.strip().lower().replace("_", " ") in BY_DESIGN else unmapped)[key] = seen[key]

    n_un = sum(unmapped.values())
    return {
        "field": label,
        "rows": len(s),
        "distinct": len(buckets),
        "unmapped rows": n_un,
        "% unmapped": round(100 * n_un / max(len(s), 1), 2),
        "unmapped values": dict(sorted(unmapped.items(), key=lambda kv: -kv[1])[:8]),
        "rejected by design": dict(sorted(by_design.items(), key=lambda kv: -kv[1])[:6]),
    }


def margin_after_migration(v):
    # 'microlobulated' is no longer a mammography margin in v2025; the migration moves it
    # to the shape field, so the value is still accounted for.
    shape, margin = P.migrate_mammography_microlobulated(None, v)
    return margin or shape


cov_rows = []

In [ ]:
# --- CBIS-DDSM ------------------------------------------------------------
from utils.preprocessing import column_cleaning_csv_reading

try:
    calc = pd.concat(
        [column_cleaning_csv_reading(str(RAW / f"cbis-ddsm/calc_case_description_{s}_set.csv")) for s in ("train", "test")]
    )
    mass = pd.concat(
        [column_cleaning_csv_reading(str(RAW / f"cbis-ddsm/mass_case_description_{s}_set.csv")) for s in ("train", "test")]
    )
    cov_rows += [
        coverage("cbis-ddsm / breast density", pd.concat([calc, mass])["breast density"], P.get_proper_breast_density),
        coverage("cbis-ddsm / assessment", pd.concat([calc, mass])["assessment"], P.map_birads),
        coverage("cbis-ddsm / mass shape", mass["mass shape"], P.get_proper_mass_shape),
        # the adapter routes margins through the microlobulated migration, which may
        # legitimately move the value into the shape field instead
        coverage("cbis-ddsm / mass margins", mass["mass margins"], margin_after_migration),
        coverage("cbis-ddsm / calc type", calc["calc type"], P.get_proper_calcification_type),
        coverage("cbis-ddsm / calc distribution", calc["calc distribution"], P.get_proper_calcification_distribution),
        coverage("cbis-ddsm / image view", pd.concat([calc, mass])["image view"], P.get_proper_exam_view),
    ]
    print("cbis-ddsm ok")
except Exception as e:
    print("cbis-ddsm skipped:", e)

In [ ]:
# --- INbreast -------------------------------------------------------------
from utils.preprocessing import csv_column_cleaning

try:
    ib = pd.read_excel(RAW / "inbreast/INbreast.xls")
    ib.columns = csv_column_cleaning(list(ib.columns))
    ib = ib.iloc[:-2]
    cov_rows += [
        coverage("inbreast / bi-rads", ib["bi-rads"], P.map_birads),
        coverage("inbreast / acr density", ib["acr"], P.get_proper_breast_density),
        coverage("inbreast / view", ib["view"], P.get_proper_exam_view),
        coverage("inbreast / laterality", ib["laterality"], P.normalize_laterality),
    ]
    print("inbreast ok")
except Exception as e:
    print("inbreast skipped:", e)

# --- breast-micro-calc ----------------------------------------------------
try:
    frames = []
    for sheet in ("Normal_cases_modified", "Suspicious_cases_modified"):
        d = pd.read_excel(RAW / "breast-micro-calc/Description.xlsx", sheet_name=sheet)
        d.columns = csv_column_cleaning(list(d.columns))
        frames.append(d)
    bmc = pd.concat(frames, ignore_index=True).replace("NOT AVAILABLE", None)
    bmc = bmc.rename(columns={"bi-rads categories for breast density": "breast density",
                              "bi-rads categories for classification": "birads",
                              "breast right/left": "laterality"})
    cov_rows += [
        coverage("breast-micro-calc / birads", bmc["birads"], P.map_birads),
        coverage("breast-micro-calc / density", bmc["breast density"], P.get_proper_breast_density),
        coverage("breast-micro-calc / laterality", bmc["laterality"], P.normalize_laterality),
    ]
    print("breast-micro-calc ok")
except Exception as e:
    print("breast-micro-calc skipped:", e)

In [ ]:
# --- BrEaST (breast-lesions-usg) -----------------------------------------
try:
    br = pd.read_excel(RAW / "breast-lesions-usg/BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx", sheet_name=0)
    br.columns = csv_column_cleaning(list(br.columns))
    br = br.replace("not applicable", None).replace("not available", None)
    cov_rows += [
        coverage("breast-lesions-usg / birads", br["birads"], P.map_birads, modality=P.ULTRASOUND),
        coverage("breast-lesions-usg / tissue composition", br["tissue composition"], P.get_proper_tissue_composition),
        coverage("breast-lesions-usg / shape", br["shape"], P.get_proper_mass_shape),
        coverage("breast-lesions-usg / margin", br["margin"], P.get_proper_mass_margin, modality=P.ULTRASOUND),
        coverage("breast-lesions-usg / echogenicity", br["echogenicity"], P.get_proper_mass_echo_pattern),
        coverage("breast-lesions-usg / posterior features", br["posterior features"], P.get_proper_posterior_features),
        coverage("breast-lesions-usg / halo", br["halo"], P.get_proper_halo),
    ]
    print("breast-lesions-usg ok")

    # 4A/4B/4C granularity actually present in the source but collapsed by the US branch
    sub = br["birads"].astype(str).str.lower().isin(["4a", "4b", "4c"]).sum()
    note("8. coverage", "breast-lesions-usg 4A/4B/4C rows collapsed to '(4) suspicious'", f"{sub} rows")
except Exception as e:
    print("breast-lesions-usg skipped:", e)

In [ ]:
# --- TOMPEI-CMMD ----------------------------------------------------------
try:
    tc = pd.read_excel(RAW / "tompei-cmmd/TOMPEI-CMMD_clinical_data_v01_20250121.xlsx",
                       sheet_name="Imaging Diagnosis Details Sheet")
    from preprocessing.tompei_cmmd import TompeiCMMD  # for the column list only
    cols = ["patient id", "laterality", "age", "classification", "exclusion reason", "breast density",
            "mass location", "mass shape", "mass margin", "mass density", "mass associated calcification",
            "mass other associated findings", "calcification location", "calcification morphology",
            "calcification distribution", "calcification clearly benign calcifications",
            "other associated findings", "other findings location", "other findings breast parenchyma",
            "other findings skin", "other findings lymph nodes", "birads", "notes",
            "mass - additional lesion location", "mass - additional lesion shape",
            "mass - additional lesion margin", "mass - additional lesion density",
            "mass - additional lesion associated calcification",
            "mass - additional lesion other associated findings",
            "calcification - additional lesion location", "calcification - additional lesion morphology",
            "calcification - additional lesion distribution",
            "calcification - additional lesion clearly benign calcifications"]
    check("8. coverage", "TOMPEI-CMMD sheet still has the 33 columns the adapter hard-codes",
          tc.shape[1] == len(cols), f"{tc.shape[1]} columns in the sheet, adapter expects {len(cols)}")
    tc = tc.iloc[1:]
    tc.columns = cols
    cov_rows += [
        coverage("tompei-cmmd / birads", tc["birads"], P.map_birads, pre=lambda v: int(v)),
        coverage("tompei-cmmd / breast density", tc["breast density"], P.get_proper_breast_density),
        coverage("tompei-cmmd / laterality", tc["laterality"], P.normalize_laterality),
        coverage("tompei-cmmd / mass shape", tc["mass shape"], P.get_proper_mass_shape),
        coverage("tompei-cmmd / mass margin", tc["mass margin"], margin_after_migration),
        coverage("tompei-cmmd / mass density", tc["mass density"], P.get_proper_mass_density),
        coverage("tompei-cmmd / calc morphology", tc["calcification morphology"], P.get_proper_calcification_type),
        coverage("tompei-cmmd / calc distribution", tc["calcification distribution"],
                 P.get_proper_calcification_distribution),
        coverage("tompei-cmmd / mass location code", tc["mass location"],
                 lambda v: P.get_value(v, P.region_mapping)),
    ]
    print("tompei-cmmd ok")
except Exception as e:
    print("tompei-cmmd skipped:", e)

In [ ]:
# --- MAMA-MIA -------------------------------------------------------------
try:
    mm = pd.read_excel(RAW / "mama-mia/clinical_and_imaging_info.xlsx")
    mm.columns = csv_column_cleaning(list(mm.columns))
    cov_rows += [
        coverage("mama-mia / breast density (fgt)", mm["breast density"], P.get_proper_mri_fgt),
        coverage("mama-mia / ethnicity", mm["ethnicity"], lambda v: P.get_value(v, P.race_mappings)),
        coverage("mama-mia / has implant", mm["has implant"], lambda v: P.get_value(v, P.yes_no_mapping)),
    ]
    print("mama-mia ok")
except Exception as e:
    print("mama-mia skipped:", e)

# --- LA-Breast ------------------------------------------------------------
try:
    la = pd.concat([column_cleaning_csv_reading(str(RAW / f"la-breast/{s}.csv")) for s in ("train", "test", "val")],
                   ignore_index=True)
    cov_rows += [
        coverage("la-breast / birads", la["birads"], P.map_birads, modality=P.MRI),
        coverage("la-breast / acr (fgt)", la["acr"], P.get_proper_mri_fgt),
        coverage("la-breast / sequence -> contrast phase", pd.Series(["d0", "d1", "d2", "d3", "d4", "d5"]),
                 P.get_proper_contrast_phase),
    ]
    print("la-breast ok")
except Exception as e:
    print("la-breast skipped:", e)

In [ ]:
# --- Advanced-MRI-Breast-Lesions -----------------------------------------
try:
    am = pd.read_excel(RAW / "advanced-mri-breast-lesions/Advanced-MRI-Breast-Lesions-DA-Clinical-Jan112024.xlsx",
                       sheet_name=0, header=1)
    am.columns = [c.replace("id#", "") for c in csv_column_cleaning(list(am.columns))]
    cov_rows += [
        coverage("advanced-mri / birads", am["birads"], P.map_birads, modality=P.MRI),
        coverage("advanced-mri / breast implants", am["breast implants"], lambda v: P.get_value(v, P.yes_no_mapping)),
    ]
    lat = pd.Series([str(v)[0] for v in am["pos1"].dropna()])
    cov_rows.append(coverage("advanced-mri / lesion laterality prefix", lat,
                             lambda v: P.get_value(v, P.LATERALITY_MAP)))
    print("advanced-mri ok")
except Exception as e:
    print("advanced-mri skipped:", e)

# --- BUS-CoT --------------------------------------------------------------
try:
    from preprocessing.bus_cot import BUSCoT
    bc = BUSCoT.construct_csv_from_json(str(RAW / "BUSCoT/DatasetFiles/BUS-Expert_dataset.json"))
    cov_rows += [
        coverage("bus-cot / birads", bc["birads"], P.map_birads, modality=P.ULTRASOUND),
        coverage("bus-cot / side", bc["side"], P.normalize_laterality),
        coverage("bus-cot / echo", bc["mass_echo"], P.get_proper_mass_echo_pattern),
    ]
    note("8. coverage", "bus-cot modality values", sorted(set(bc["modality"].dropna().astype(str)))[:15])
    print("bus-cot ok")
except Exception as e:
    print("bus-cot skipped:", e)

In [ ]:
# --- EMBED ----------------------------------------------------------------
try:
    from preprocessing.embed import Embed
    emb = pd.read_csv(RAW / "embed/EMBED_OpenData_clinical.csv", low_memory=False,
                      usecols=["asses", "tissueden", "massshape", "massmargin", "massdens",
                               "calcfind", "calcdistri", "side", "location", "ETHNICITY_DESC"])
    legend = pd.read_csv(RAW / "embed/AWS_Open_Data_Clinical_Legend.csv")
    legend_dict = {}
    for _, r in legend.iterrows():
        if P.isna_v2(r["Code"]) or P.isna_v2(r["Meaning"]):
            continue
        legend_dict.setdefault(r["Header in export"], {})[r["Code"].strip()] = r["Meaning"].strip().lower()

    def decode(col, val):
        if P.isna_v2(val):
            return None
        parts = [legend_dict.get(col, {}).get(v.strip(), "") for v in str(val).split(",") if v.strip()]
        joined = ",".join([p for p in parts if p])
        return joined or None

    cov_rows += [
        # 'A' (needs additional imaging) and 'X' (no assessment) are filtered out upstream
        coverage("embed / asses (birads)", emb.loc[~emb["asses"].isin(["A", "X"]), "asses"],
                 P.map_birads, pre=lambda v: decode("asses", v)),
        coverage("embed / tissueden", emb["tissueden"].dropna().apply(lambda x: str(int(float(x)))),
                 lambda v: P.get_proper_breast_density(legend_dict["tissueden"].get(v))),
        coverage("embed / massshape", emb["massshape"].dropna().apply(lambda v: decode("massshape", v)),
                 P.get_proper_mass_shape),
        coverage("embed / massmargin", emb["massmargin"].dropna().apply(lambda v: decode("massmargin", v)),
                 P.get_proper_mass_margin, modality=P.MAMMOGRAPHY),
        coverage("embed / massdens", emb["massdens"].dropna().apply(lambda v: decode("massdens", v)),
                 P.get_proper_mass_density),
        coverage("embed / calcfind", emb["calcfind"].dropna().apply(lambda v: decode("calcfind", v)),
                 P.get_proper_calcification_type),
        coverage("embed / calcdistri", emb["calcdistri"].dropna().apply(lambda v: decode("calcdistri", v)),
                 P.get_proper_calcification_distribution),
        # the "declined / not recorded" surfaces are turned into UNKNOWN by the adapter
        coverage("embed / ethnicity", emb["ETHNICITY_DESC"], lambda v: P.get_value(v, P.race_mappings)),
    ]
    note("8. coverage", "embed 'side' decoded values", sorted(set(emb["side"].dropna().apply(lambda v: decode("side", v)).dropna()))[:10])
    print("embed ok")
except Exception as e:
    print("embed skipped:", e)

In [ ]:
cov = pd.DataFrame(cov_rows).sort_values("% unmapped", ascending=False).reset_index(drop=True)
display(cov)

worst = cov[cov["% unmapped"] > 5]
check(
    "8. coverage",
    "no mapped field loses more than 5% of its rows to an unrecognised value",
    worst.empty,
    worst[["field", "% unmapped", "unmapped values"]].to_dict("records") if not worst.empty else "",
)

by_design = cov[cov["rejected by design"].apply(bool)]
display(by_design[["field", "rows", "rejected by design"]].reset_index(drop=True))
note(
    "8. coverage",
    "values rejected on purpose",
    "these are real findings in the source that the field they live in cannot express "
    "(architectural distortion / asymmetry / lymph node in a 'mass shape' column, ...). "
    "utils.preprocessing.get_proper_finding_type exists to route them, see the next cell.",
)

### 8b. Non-mass findings that end up typed as a mass

`get_proper_finding_type` is documented as the router to call *before* the mass mappers:
a row whose `mass shape` reads `architectural_distortion` is not a mass with an unmappable
shape. No adapter calls it, so those rows still take the mass branch.

In [ ]:
S = "8. coverage"
import inspect
import pkgutil

import preprocessing as _pkg

unused = []
for name in ["get_proper_finding_type", "get_proper_asymmetry_type", "get_proper_orientation",
             "get_proper_non_mass_distribution", "get_proper_non_mass_echo_pattern",
             "get_proper_bpe_level", "get_proper_bpe_symmetry", "get_proper_t2_signal_intensity",
             "get_proper_early_phase", "get_proper_delayed_phase", "get_proper_vascularity",
             "get_proper_elasticity", "get_proper_echogenic_feature",
             "get_proper_glandular_tissue_component", "get_proper_mass_internal_enhancement",
             "get_proper_nme_internal_enhancement"]:
    used_by = []
    for mod in pkgutil.iter_modules(_pkg.__path__):
        src = (Path("preprocessing") / f"{mod.name}.py").read_text()
        if name in src:
            used_by.append(mod.name)
    if not used_by:
        unused.append(name)
display(pd.DataFrame({"mapper defined but never called by an adapter": unused}))
note(S, "unused lexicon surface", f"{len(unused)} of the v2025 mappers are not reachable from any adapter")

# What the mass branch does with a non-mass finding, using CBIS-DDSM as the example.
try:
    d = pd.concat([column_cleaning_csv_reading(str(RAW / f"cbis-ddsm/mass_case_description_{s}_set.csv"))
                   for s in ("train", "test")])
    non_mass = d[d["mass shape"].apply(lambda v: P.get_proper_mass_shape(v) is None and not P.isna_v2(v))]
    routed = non_mass["mass shape"].apply(lambda v: P.get_proper_finding_type(v, P.MAMMOGRAPHY))
    tbl = pd.DataFrame({"mass shape value": non_mass["mass shape"], "get_proper_finding_type says": routed})
    display(tbl.value_counts().to_frame("rows").head(10))
    unrouted = routed[routed.isna()]
    check(
        S,
        "every non-mass value in the CBIS 'mass shape' column is routed to a finding type",
        unrouted.empty,
        f"{len(non_mass)} of {len(d)} rows carry a non-mass descriptor; "
        f"{len(unrouted)} of them could not be routed",
    )

    # the adapter must take the routed branch rather than Lesion(type=MASS)
    from preprocessing.cbis_ddsm import CbisDDSM
    src = inspect.getsource(CbisDDSM.process_row)
    check(
        S,
        "CbisDDSM.process_row routes the finding before assuming a mass",
        "get_proper_finding_type" in src,
        "otherwise architectural distortion / lymph node / asymmetry rows are emitted "
        "as a mass with no descriptors",
    )

    from preprocessing.embed import Embed
    check(
        S,
        "Embed.construct_lesion routes the finding before assuming a mass",
        "get_proper_finding_type" in inspect.getsource(Embed.construct_lesion),
    )
except Exception as e:
    print("cbis finding-type check skipped:", e)

In [ ]:
# The mappers log the first sighting of each unrecognised value; this is what they saw.
from utils.preprocessing import _UNMAPPED_COUNTS

unmapped = pd.DataFrame(
    [{"field": f, "value": v, "times": c} for (f, v), c in _UNMAPPED_COUNTS.most_common(60)]
)
print(f"{len(_UNMAPPED_COUNTS)} distinct (field, value) pairs were rejected during this audit")
display(unmapped)

## 9. Mask alignment on real image/mask pairs

The single most damaging silent failure would be a mask that does not sit on its lesion.
Each pair below is read from source, pushed through the real pipelines, and drawn with the
mask contour on top of the processed image.

In [ ]:
S = "9. masks"
panels = []   # (title, processed image, processed mask)


def run_pipelines(img, mask):
    pi, pm = img, mask
    for step in cfg.processing_pipeline:
        pi = step(pi)
    for step in cfg.segmentation_pipeline:
        pm = step(pm)
    return pi, pm


def record(title, img, mask):
    pi, pm = run_pipelines(img, mask)
    panels.append((title, pi, pm))
    frac = float((pm > 0).mean())
    inside = float(pi[pm > 0].mean()) if (pm > 0).any() else 0.0
    outside = float(pi[pm == 0].mean()) if (pm == 0).any() else 0.0
    check(S, f"{title}: mask is non-empty and plausibly sized",
          0 < frac < 0.6, f"mask covers {100 * frac:.1f}% of the frame; "
          f"mean intensity inside={inside:.0f} vs outside={outside:.0f}")
    return pi, pm

In [ ]:
# --- CBIS-DDSM: the ROI folder holds a crop and a full-frame mask ---------
try:
    d = column_cleaning_csv_reading(str(RAW / "cbis-ddsm/mass_case_description_train_set.csv"))
    row = d.iloc[0]
    img_dir = RAW / "cbis-ddsm/jpeg" / row["image file path"].split("/")[-2]
    roi_dir = RAW / "cbis-ddsm/jpeg" / row["roi mask file path"].split("/")[-2]
    full = read_breast_image(str(sorted(img_dir.glob("*.jpg"))[0]))
    cands = {p.name: read_breast_image(str(p)) for p in sorted(roi_dir.glob("*.jpg"))}
    note(S, "cbis-ddsm ROI folder contents",
         {k: v.shape for k, v in cands.items()} | {"full image": full.shape})
    picked = [k for k, v in cands.items() if v.shape == full.shape]
    check(S, "exactly one ROI file matches the full-image geometry", len(picked) == 1, picked)
    if picked:
        record("cbis-ddsm", full, cands[picked[0]])
except Exception as e:
    print("cbis-ddsm mask check skipped:", e)

In [ ]:
# --- INbreast: polygon ROIs from the plist XML ---------------------------
try:
    from preprocessing.inbreast import Inbreast

    xmls = sorted((RAW / "inbreast/AllXML").glob("*.xml"))
    done = False
    for xml in xmls:
        file_id = xml.stem
        dcm = next(iter(sorted((RAW / "inbreast/AllDICOMs").glob(f"{file_id}_*.dcm"))), None)
        if dcm is None:
            continue
        img = convert_dcm_image(str(dcm))
        mask = Inbreast.load_inbreast_mask(str(xml), img.shape)
        if mask is None or mask.max() == 0:
            continue
        record(f"inbreast {file_id}", img, mask)
        done = True
        break
    check(S, "inbreast XML polygons rasterise onto the DICOM frame", done)
except Exception as e:
    print("inbreast mask check skipped:", e)

In [ ]:
# --- BrEaST ultrasound: image + tumour mask ------------------------------
try:
    br = pd.read_excel(RAW / "breast-lesions-usg/BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx", sheet_name=0)
    br.columns = csv_column_cleaning(list(br.columns))
    root = RAW / "breast-lesions-usg/BrEaST-Lesions_USG-images_and_masks"
    r = br[br["mask tumor filename"].notna()].iloc[0]
    img = read_breast_image(str(root / r["image filename"]))
    mask = read_breast_image(str(root / r["mask tumor filename"]))
    check(S, "BrEaST image and mask share a frame", img.shape == mask.shape, f"{img.shape} vs {mask.shape}")
    record(f"breast-lesions-usg {r['case id']}", img, mask)
except Exception as e:
    print("breast-lesions-usg mask check skipped:", e)

In [ ]:
# --- EMBED: ROI boxes in original DICOM coordinates, resized to the PNG --
try:
    from utils.preprocessing import create_segmentation_mask, resize_breast_image

    md_cols = ["anon_dicom_path", "num_roi", "ROI_coords", "match_level"]
    emb_md = pd.read_csv(RAW / "embed/EMBED_OpenData_metadata_reduced.csv", low_memory=False, usecols=md_cols)
    emb_md = emb_md[emb_md["num_roi"] > 0].copy()
    emb_md["image_id"] = emb_md["anon_dicom_path"].apply(lambda x: os.path.basename(x)[:-4])
    sizes = pd.read_csv(RAW / "embed/image_sizes.csv",
                        names=["image path", "original width", "original height", "resized width", "resized height"])
    sizes["image_id"] = sizes["image path"].apply(lambda x: os.path.basename(x)[:-4])
    emb_md = emb_md.merge(sizes, on="image_id", how="inner")

    import ast
    hit = None
    for _, r in emb_md.iterrows():
        p = RAW / "embed/imgs" / f"{r['image_id']}.png"
        if p.exists():
            hit = (r, p)
            break
    if hit is None:
        raise FileNotFoundError("no EMBED png on disk matches a row with an ROI")

    r, p = hit
    img = read_breast_image(str(p))
    rois = tuple(tuple(int(v) for v in roi) for roi in ast.literal_eval(str(r["ROI_coords"]))
                 if isinstance(roi, (list, tuple)) and len(roi) == 4)
    mask = create_segmentation_mask(height=r["original height"], width=r["original width"], rois=rois)
    mask = resize_breast_image(mask, (r["resized width"], r["resized height"]))
    check(S, "EMBED png on disk is the 'resized' geometry from image_sizes.csv",
          img.shape == (r["resized height"], r["resized width"]),
          f"png {img.shape} vs csv ({r['resized height']}, {r['resized width']})")
    record(f"embed {r['image_id'][-8:]}", img, mask)
    note(S, "EMBED roi + match_level of this row", f"{r['ROI_coords']}  match_level={r['match_level']!r}")
except Exception as e:
    print("embed mask check skipped:", e)

In [ ]:
# --- MAMA-MIA: expert 3D segmentation, central lesion slice --------------
try:
    seg_dir = RAW / "mama-mia/segmentations/expert"
    seg_file = next(iter(sorted(seg_dir.glob("*.nii.gz"))), None)
    pid = seg_file.name.replace(".nii.gz", "")
    vol_files = sorted((RAW / "mama-mia/images" / pid).glob("*.nii.gz"))
    seg = read_nii_gz_images(str(seg_file))
    vol = read_nii_gz_images(str(vol_files[0]))
    check(S, "MAMA-MIA image and segmentation have the same slice count",
          vol.shape == seg.shape, f"{vol.shape} vs {seg.shape}")
    lesion_slices = [i for i in range(seg.shape[0]) if seg[i].sum() > 0]
    central = lesion_slices[len(lesion_slices) // 2]
    note(S, f"mama-mia {pid}", f"{len(lesion_slices)} lesion-bearing slices of {seg.shape[0]}; central={central}")
    record(f"mama-mia {pid} z={central}", vol[central], seg[central])
except Exception as e:
    print("mama-mia mask check skipped:", e)

In [ ]:
# --- OASBUD: RF frame + its ROI -----------------------------------------
try:
    data = read_mat_images(str(RAW / "oasbud/OASBUD.mat"))
    rec = data[0]
    rf, roi = rec["rf1"], rec["roi1"]
    bmode = process_us_from_mat(rf)
    check(S, "OASBUD B-mode and ROI share a frame", bmode.shape == roi.shape, f"{bmode.shape} vs {roi.shape}")
    record("oasbud (as shipped, axis=-1)", bmode, roi)
except Exception as e:
    print("oasbud mask check skipped:", e)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, (title, img, mask) in zip(axes.ravel(), panels):
    show(ax, img, mask, title=title)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
fig.suptitle("Processed image with the processed mask contour (red)")
plt.tight_layout()
plt.show()

## 10. OASBUD — RF demodulation axis

Envelope detection has to run along the **depth (axial)** direction, because that is where
the transducer carrier lives. OASBUD frames are `(samples, lines)`, so depth is `axis=0`;
demodulating across scan lines does not recover an envelope.

In [ ]:
from scipy.signal import hilbert

S = "10. oasbud"
try:
    data = read_mat_images(str(RAW / "oasbud/OASBUD.mat"))
    print("record fields:", data.dtype.names)
    check(S, "the tuple unpacking in Oasbud.process_info matches the .mat field order",
          data.dtype.names == ("id", "rf1", "rf2", "roi1", "roi2", "birads", "class"), data.dtype.names)

    rf = data[0]["rf1"].astype(np.float32)
    print("rf shape (samples, lines):", rf.shape)

    # Which axis carries the RF carrier? Take the mean power spectrum along each axis.
    rows = []
    for rec in range(6):
        r = data[rec]["rf1"].astype(np.float32)
        for ax, name in [(0, "axis 0 (rows)"), (1, "axis 1 (cols)")]:
            F = (np.abs(np.fft.rfft(r - r.mean(), axis=ax)) ** 2).mean(axis=1 - ax)
            f = np.fft.rfftfreq(r.shape[ax])
            k = f > 0.02
            rows.append({"record": rec, "axis": name, "n": r.shape[ax],
                         "peak freq (fraction of fs)": round(float(f[k][np.argmax(F[k])]), 3)})
    spec = pd.DataFrame(rows).pivot(index="record", columns="axis", values="peak freq (fraction of fs)")
    display(spec)

    carrier_ax0 = spec["axis 0 (rows)"].between(0.05, 0.25).mean()
    note(S, "carrier location",
         f"{100 * carrier_ax0:.0f}% of sampled records show a 0.05-0.25*fs carrier on axis 0 "
         "(~4.7 MHz at a 40 MHz sampling rate); axis 1 shows none")

    import inspect

    from utils.preprocessing import process_us_from_mat as _pufm
    fn = _pufm
    while hasattr(fn, "__wrapped__"):
        fn = fn.__wrapped__
    default_axis = inspect.signature(fn).parameters["axis"].default
    check(S, "process_us_from_mat demodulates along the depth axis by default",
          default_axis == 0, f"axis default = {default_axis}")

    def reference(rf_frame, ax):
        env = np.abs(hilbert(np.asarray(rf_frame, dtype=np.float32), axis=ax))
        peak = env.max()
        floor = peak * 10 ** (-50 / 20)
        c = 20 * np.log10(np.maximum(env, floor) / peak)
        return np.rint(np.clip((c + 50) / 50, 0, 1) * 65535).astype(np.uint16)

    shipped = process_us_from_mat(data[0]["rf1"])
    check(S, "the shipped B-mode matches an axial demodulation, not a lateral one",
          np.array_equal(shipped, reference(rf, 0)) and not np.array_equal(shipped, reference(rf, -1)))

    roi = np.asarray(data[0]["roi1"]) > 0
    inside, outside = float(shipped[roi].mean()), float(shipped[~roi].mean())
    check(S, "the annotated lesion is hypoechoic relative to its surroundings",
          inside < outside, f"mean inside ROI={inside:.0f} vs outside={outside:.0f}")

    check(S, "log compression survives a zero-valued envelope sample",
          np.isfinite(process_us_from_mat(np.pad(rf[:64, :64], 8)).astype(float)).all())
except Exception as e:
    print("oasbud spectrum check skipped:", e)

In [ ]:
try:
    def bmode(rf, axis):
        env = np.abs(hilbert(rf, axis=axis))
        c = 20 * np.log10(np.maximum(env, 1e-9) / env.max())
        return np.clip(c, -50, 0)

    fig, axes = plt.subplots(1, 3, figsize=(14, 6))
    show(axes[0], bmode(rf, -1), data[0]["roi1"], title="lateral: hilbert(axis=-1)  [wrong]")
    show(axes[1], bmode(rf, 0), data[0]["roi1"], title="as shipped: hilbert(axis=0)  [depth]")
    axes[2].plot(rf[400:520, rf.shape[1] // 2], lw=0.8)
    axes[2].set_title("one A-line (axis 0): the RF carrier oscillation", fontsize=8)
    fig.suptitle("OASBUD envelope detection -- axis choice")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("oasbud figure skipped:", e)

## 11. EMBED — `match_level` filter and ROI coordinate order

`ROI_coords` and `match_level` are both per-ROI lists serialised as text, so the primary /
secondary grading has to be applied box by box. A scalar `match_level != 1` test is true for
every row in the collection and would clear every ROI.

In [ ]:
S = "11. embed"
try:
    ml = pd.read_csv(RAW / "embed/EMBED_OpenData_metadata_reduced.csv", low_memory=False,
                     usecols=["match_level", "ROI_coords", "num_roi"])
    print("match_level dtype:", ml["match_level"].dtype)
    display(ml["match_level"].value_counts(dropna=False).head(8).to_frame("rows"))

    from preprocessing.embed import Embed

    note(S, "a scalar comparison would fail",
         f"{int((ml['match_level'] == 1).sum())} rows compare equal to the integer 1 -- "
         "the grading only means something once the list is parsed")

    # unit behaviour of the element-wise filter
    cases = {
        ("((1,2,3,4),)", "[1]"): ((1, 2, 3, 4),),
        ("((1,2,3,4),)", "[2]"): None,
        ("((1,2,3,4),(5,6,7,8))", "[1, 2]"): ((1, 2, 3, 4),),
        ("((1,2,3,4),(5,6,7,8))", "[2, 2]"): None,
        ("()", "[]"): None,
        ("((1,2,3,4),)", "[]"): None,
        ("((1,2,3,4),(5,6,7,8))", "[1]"): None,   # length mismatch -> cannot tell
    }
    got = {k: Embed.keep_primary_rois(*k) for k in cases}
    display(pd.DataFrame({"roi_coords": [k[0] for k in cases], "match_level": [k[1] for k in cases],
                          "kept": [got[k] for k in cases], "expected": [cases[k] for k in cases]}))
    check(S, "keep_primary_rois keeps exactly the level-1 boxes", got == cases,
          {k: (got[k], cases[k]) for k in cases if got[k] != cases[k]})

    # on the real collection
    before = ml["ROI_coords"].apply(Embed.count_rois).sum()
    kept = ml.apply(lambda r: Embed.keep_primary_rois(r["ROI_coords"], r["match_level"]), axis=1)
    after = kept.apply(Embed.count_rois).sum()
    check(S, "primary ROIs survive the filter on the real metadata", after > 0,
          f"{after} of {before} ROI boxes kept, across {int(kept.notna().sum())} images")

    # "()" is the literal empty value in the source; the filter normalises it to None
    empties = int((ml["ROI_coords"].astype(str).str.strip() == "()").sum())
    check(S, "an absent ROI is normalised to None so isna() is a valid emptiness test",
          bool(kept[ml["ROI_coords"].astype(str).str.strip() == "()"].isna().all()),
          f'{empties} source rows carry the literal string "()" rather than NaN')
except Exception as e:
    print("embed match_level check skipped:", e)

In [ ]:
# ROI coordinate order, decided by in-bounds counting against the true image sizes.
try:
    md2 = pd.read_csv(RAW / "embed/EMBED_OpenData_metadata_reduced.csv", low_memory=False,
                      usecols=["anon_dicom_path", "num_roi", "ROI_coords"])
    md2 = md2[md2["num_roi"] > 0].copy()
    md2["image_id"] = md2["anon_dicom_path"].apply(lambda x: os.path.basename(x)[:-4])
    m = md2.merge(sizes, on="image_id", how="inner")

    yx = xy = n = 0
    for _, r in m.iterrows():
        try:
            rois = ast.literal_eval(str(r["ROI_coords"]))
        except Exception:
            continue
        for roi in rois:
            if not isinstance(roi, (list, tuple)) or len(roi) != 4 or any(isinstance(v, (list, tuple)) for v in roi):
                continue
            _, _, c, d = [float(v) for v in roi]
            n += 1
            yx += c <= r["original height"] and d <= r["original width"]
            xy += c <= r["original width"] and d <= r["original height"]
    print(f"{n} ROI boxes checked")
    print(f"  in bounds as (ymin, xmin, ymax, xmax): {100 * yx / n:.1f}%")
    print(f"  in bounds as (xmin, ymin, xmax, ymax): {100 * xy / n:.1f}%")
    check(S, "create_segmentation_mask's (ymin, xmin, ymax, xmax) reading matches EMBED",
          yx > xy and yx / n > 0.99, f"{100 * yx / n:.1f}% vs {100 * xy / n:.1f}%")
except Exception as e:
    print("embed roi order check skipped:", e)

In [ ]:
# The mask is built binary, then resized with the *image* resizer (bilinear).
try:
    demo = create_segmentation_mask(height=200, width=200, rois=((80, 80, 120, 120),))
    small = resize_breast_image(demo, (60, 60))
    note(S, "mask resize interpolation",
         f"resize_breast_image on a binary mask yields {len(np.unique(small))} distinct values "
         f"{np.unique(small)[:6]}; binarize_mask(>0) afterwards grows the mask by every "
         "partially-covered pixel. resize_mask (INTER_NEAREST) is the mask-safe resizer.")
    grown = (binarize_mask(small).mean() - resize_mask(demo, (60, 60)).mean()) / max(resize_mask(demo, (60, 60)).mean(), 1e-9)
    check(S, "bilinear+threshold vs nearest changes the mask area by <2%", abs(grown) < 0.02,
          f"area difference {100 * grown:+.1f}%")
except Exception as e:
    print("embed resize check skipped:", e)

## 12. LA-Breast — the ROI rectangle and the clinical-group sampling

`draw_rectangle_on_image` copies its input (section 2), so its return value has to be bound
or the mask stays empty and every row is dropped for having no usable ROI. The dedup step
also has to cope with the missing `breast density` on a fifth of the rows.

In [ ]:
S = "12. la-breast"
try:
    la = column_cleaning_csv_reading(str(RAW / "la-breast/train.csv"))
    r = la.iloc[0]
    stem = str(r["d0"]).split("/")[-1].split(".")[0]
    patient = str(r["patient"]).replace("Breast_", "")
    hits = sorted((RAW / "la-breast/imgs").glob(f"*/*/{patient}*{stem}*.tiff"))
    print("row:", patient, "roi:", r["roi"], "d0:", r["d0"])
    print("matching tiffs:", [h.name for h in hits][:3])

    img = read_breast_image(str(hits[0])) if hits else np.zeros((512, 512), np.uint16)
    h, w = img.shape
    dx, dy, cx, cy = r["distancia x d0"], r["distancia y d0"], r["centro x d0"], r["centro y d0"]
    x1, y1 = int(cx - dx / 2), int(cy - dy / 2)
    x2, y2 = int(cx + dx / 2), int(cy + dy / 2)
    print(f"image {img.shape}, box (x1,y1)-(x2,y2) = ({x1},{y1})-({x2},{y2})")

    import inspect

    from preprocessing.la_breast import LABreast

    src = inspect.getsource(LABreast.create_and_save_segmentation)
    check(S, "create_and_save_segmentation binds the drawn rectangle",
          "mask = draw_rectangle_on_image(" in src,
          "draw_rectangle_on_image returns a copy; discarding it leaves an empty mask and "
          "save_segmentation_set then rejects the row")

    mask = draw_rectangle_on_image(np.zeros(img.shape, np.uint8), x1, y1, x2, y2)
    check(S, "the ROI rectangle reaches the mask", mask is not None and mask.max() > 0,
          f"{int((mask > 0).sum())} pixels set")
    check(S, "the box lands inside the image frame", 0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h,
          f"frame is (h={h}, w={w})")

    processed = mask
    for step in cfg.segmentation_pipeline:
        processed = step(processed)
    check(S, "the ROI survives the segmentation pipeline", processed.max() > 0,
          f"{int((processed > 0).sum())} of {processed.size} pixels at {processed.shape}")

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    show(axes[0], img, binarize_mask(mask), title="ROI box on the raw slice")
    pi = img
    for step in cfg.processing_pipeline:
        pi = step(pi)
    show(axes[1], pi, processed, title="after the image / mask pipelines")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("la-breast check skipped:", e)

## 12b. Advanced-MRI — which series may emit an image, and with what phase

The lesion coordinates in the clinical sheet are **axial** slice positions, so a sagittal
series cannot be matched against them. `TRAM` is a derived `PALETTE COLOR` map whose stored
values are colour-table indices. And only a dynamic multi-phase series has contrast phases
to enumerate — labelling a T2 FSE image "pre-contrast" because it is first in its slice
group is simply false.

In [ ]:
S = "12b. advanced-mri"
try:
    import pydicom as dicom

    from preprocessing.advanced_mri_lesions import AdvancedMRILesions, raw_imgs_path

    meta = column_cleaning_csv_reading(str(RAW / "advanced-mri-breast-lesions/metadata.csv")).reset_index()
    cols = list(meta.columns[1:])
    cols.insert(cols.index("file size"), P.UNKNOWN)
    meta.columns = cols
    subject = meta[meta["modality"] == "MR"]["subject id"].iloc[0]
    series = meta[(meta["modality"] == "MR") & (meta["subject id"] == subject)]

    adapter = AdvancedMRILesions.__new__(AdvancedMRILesions)
    rows = []
    for _, srow in series.iterrows():
        folder = os.path.join(raw_imgs_path, str(srow["file location"])[2:])
        files = sorted(Path(folder).glob("*"))
        if not files:
            continue
        ds = dicom.dcmread(str(files[0]), stop_before_pixels=True)
        kept = adapter.collect_series_instances(folder, srow["series description"])
        rows.append({
            "series": srow["series description"],
            "orientation": adapter.slice_orientation(ds),
            "photometric": str(getattr(ds, "PhotometricInterpretation", "")),
            "files": len(files),
            "instances kept": len(kept),
            "contrast phase": sorted({str(e.get("contrast_phase")) for e in kept}) if kept else [],
        })
    tbl = pd.DataFrame(rows)
    display(tbl)

    sag = tbl[tbl["orientation"] == "sagittal"]
    check(S, "sagittal series emit no instances", (sag["instances kept"] == 0).all(),
          sag[["series", "instances kept"]].to_dict("records"))

    palette = tbl[tbl["photometric"].str.upper().str.startswith("PALETTE")]
    check(S, "PALETTE COLOR series (TRAM) emit no instances", (palette["instances kept"] == 0).all(),
          palette[["series", "instances kept"]].to_dict("records"))

    t2 = tbl[tbl["series"].str.lower().str.contains("t2") & (tbl["instances kept"] > 0)]
    check(S, "a T2 series is not labelled as a contrast phase",
          all("pre-contrast" not in p and "post-contrast" not in p
              for phases in t2["contrast phase"] for p in phases),
          t2[["series", "contrast phase"]].to_dict("records"))

    multi = tbl[tbl["series"].str.lower().str.replace(" ", "").str.contains("multiphase")
                & (tbl["instances kept"] > 0)]
    check(S, "the dynamic multi-phase series does enumerate contrast phases",
          all(len(phases) > 1 for phases in multi["contrast phase"]),
          multi[["series", "contrast phase"]].to_dict("records"))

    everything = [p for phases in tbl["contrast phase"] for p in phases if p != "None"]
    unmapped = [p for p in set(everything) if P.get_proper_contrast_phase(p) is None]
    check(S, "every emitted contrast-phase label is in the lexicon", not unmapped, unmapped)
except Exception as e:
    check(S, "advanced-mri series routing", False, repr(e))

In [ ]:
# convert_dcm_image must refuse a palette-colour image outright rather than read
# the colour-table indices as intensities.
S = "12b. advanced-mri"
try:
    tram = None
    for _, srow in series.iterrows():
        folder = Path(raw_imgs_path) / str(srow["file location"])[2:]
        files = sorted(folder.glob("*"))
        if not files:
            continue
        ds = dicom.dcmread(str(files[0]), stop_before_pixels=True)
        if str(getattr(ds, "PhotometricInterpretation", "")).upper().startswith("PALETTE"):
            tram = files[0]
            break
    if tram is None:
        note(S, "no PALETTE COLOR series on disk", "skipping the reader guard check")
    else:
        check(S, "convert_dcm_image refuses a PALETTE COLOR image",
              convert_dcm_image(str(tram)) is None,
              f"{tram.parent.name}: reading palette indices as grey levels is meaningless")
except Exception as e:
    check(S, "convert_dcm_image palette guard", False, repr(e))

In [ ]:
# _sample_one_row_per_clinical_group used to raise
#   KeyError: ('Breast_Mri_100', '(1) negative', nan)
# because GroupBy.sample cannot index a group whose key contains NaN, and
# `breast density` is missing on a fifth of the rows.
S = "12. la-breast"
try:
    from preprocessing.la_breast import LABreast

    la = pd.concat([column_cleaning_csv_reading(str(RAW / f"la-breast/{s}.csv"))
                    for s in ("train", "test", "val")], ignore_index=True)
    la["birads"] = la["birads"].apply(lambda v: P.map_birads(v, source="LA-Breast", modality=P.MRI))
    la["birads"] = la["birads"].apply(P.collapse_known_biopsy_proven)
    la = la.rename(columns={"acr": "breast density"})
    missing_density = int(la["breast density"].isna().sum())

    sampler = LABreast.__new__(LABreast)
    dedup = sampler._sample_one_row_per_clinical_group(la, random_state=42)

    groups = la.assign(
        _d=la["breast density"].apply(lambda v: "<missing>" if P.isna_v2(v) else str(v))
    ).groupby(["patient", "birads", "_d"], dropna=False).ngroups

    check(S, "clinical-group sampling survives a missing breast density",
          len(dedup) == groups,
          f"{missing_density} of {len(la)} rows have no ACR density; "
          f"{len(la)} rows -> {len(dedup)} representatives for {groups} groups")
    check(S, "every patient keeps at least one row",
          dedup["patient"].nunique() == la["patient"].nunique(),
          f"{dedup['patient'].nunique()} of {la['patient'].nunique()} patients")
    check(S, "exactly one row per clinical group",
          dedup.groupby(["patient", "birads"], dropna=False).size().max() <= la["roi"].nunique())
except Exception as e:
    check(S, "clinical-group sampling survives a missing breast density", False, repr(e))

## 13. Report JSON construction

In [ ]:
from dtos.json_report_context import ExamContext, JSONReportContext, PatientContext
from dtos.json_report_findings import (
    Assessment,
    BreastFindings,
    CalcificationAbnormality,
    JSONReportFindings,
    Lesion,
    LesionLocation,
    MassAbnormality,
)

S = "13. report json"

full = JSONReportFindings(
    breast=BreastFindings(density=P.get_proper_breast_density("c")),
    lesion=Lesion(
        location=LesionLocation(**P.get_proper_location("10 o'clock", "right")),
        type=P.MASS,
        mass_details=MassAbnormality(shape="irregular", margin="spiculated"),
    ),
    assessment=Assessment(birads=P.get_proper_birads(5)),
)
print(full.get_string())
parsed = json.loads(full.get_string())
check(S, "findings JSON is valid and carries breast/lesion/assessment",
      set(parsed) == {"breast", "lesion", "assessment"}, list(parsed))
check(S, "None descriptors are stripped rather than emitted as null",
      "coarse location" not in parsed["lesion"]["location"], parsed["lesion"]["location"])

empty_desc = JSONReportFindings(lesion=Lesion(type=None, mass_details=MassAbnormality()))
check(S, "a lesion with no usable descriptor degrades to PRESENT, not '{}'",
      json.loads(empty_desc.get_string())["lesion"] == P.PRESENT,
      json.loads(empty_desc.get_string()))

no_lesion = JSONReportFindings(lesion=P.NOT_PRESENT, assessment=Assessment(birads=P.BIRADS_MAPPING[1]))
check(S, "NOT_PRESENT states the negative explicitly",
      json.loads(no_lesion.get_string())["lesion"] == P.NOT_PRESENT)

unknown = JSONReportFindings(assessment=Assessment(birads=P.BIRADS_MAPPING[2]))
check(S, "lesion=None omits the key entirely (unknown != absent)",
      "lesion" not in json.loads(unknown.get_string()), json.loads(unknown.get_string()))

try:
    Lesion(type="mass", calcification_details=CalcificationAbnormality(type="typically benign"))
    ok = False
except AssertionError:
    ok = True
check(S, "a mass lesion cannot carry calcification details", ok)

ctx = JSONReportContext(
    patient_context=PatientContext(age=P.bin_age(54), has_implants="no"),
    exam_context=ExamContext(modality=P.MAMMOGRAPHY, laterality="right", view=P.CRANIAL_CAUDAL),
)
print(ctx.get_string())
check(S, "context JSON keys are sanitised to spaced lower case",
      "has implants" in json.loads(ctx.get_string())["patient"], json.loads(ctx.get_string()))
check(S, "age is binned, never emitted as an exact age",
      json.loads(ctx.get_string())["patient"]["age"] == "50-59")

In [ ]:
# Age binning boundaries
S = "13. report json"
bins = {v: P.bin_age(v) for v in [0, 39, 40, 49, 50, 59, 60, 69, 70, 79, 80, 120, "45", 45.7, None, np.nan, "abc"]}
display(pd.DataFrame({"input": [repr(k) for k in bins], "bin": list(bins.values())}))
check(S, "age bins are contiguous and closed",
      [bins[v] for v in (39, 40, 49, 50, 79, 80)] == ["<40", "40-49", "40-49", "50-59", "70-79", "80+"])
check(S, "unusable ages return None", bins[None] is None and bins["abc"] is None)
check(S, "sanitize_age rejects impossible ages",
      P.sanitize_age(-3) is None and P.sanitize_age(200) is None and P.sanitize_age(54) == "54")

## 14. Summary

In [ ]:
summary = pd.DataFrame(RESULTS)
counts = summary["status"].value_counts()
print(counts.to_string())
print()
failed = summary[summary["status"] == "FAIL"]
if failed.empty:
    print("No failing checks.")
else:
    print("FAILING CHECKS")
    for _, r in failed.iterrows():
        print(f"\n[{r['section']}] {r['check']}\n    {r['detail']}")

In [ ]:
display(summary[summary["status"] == "FAIL"][["section", "check", "detail"]].reset_index(drop=True))

In [ ]:
display(summary[summary["status"] == "INFO"][["section", "check", "detail"]].reset_index(drop=True))

In [ ]:
summary.to_csv("logs/preprocessing_audit_results.csv", index=False)
print("wrote logs/preprocessing_audit_results.csv")
summary.groupby(["section", "status"]).size().unstack(fill_value=0)